# SparkClient: Batch Job with S3A Object Storage (SeaweedFS)

This notebook demonstrates how to submit and manage a Spark batch job whose source script is stored in an S3-compatible object store (SeaweedFS) using the `s3a://` protocol.

It covers:
- Configuring S3A credentials and endpoints for in-cluster object storage.
- Setting Hadoop-AWS and modular AWS SDK v2 dependencies.
- Submitting the `FileJob` using an `s3a://` URI.
- Monitoring job lifecycle and retrieving driver execution logs.
- Cleaning up the completed job.

## 1. Imports and Client Setup

Initialize the `SparkClient` with the target Kubernetes namespace.

In [ ]:
import os

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import FileJob, SparkClient, SparkJobStatus

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

## 2. Configure S3A & Spark Settings

Define the S3 endpoint, credentials, and S3A filesystem parameters.

In [ ]:
s3_endpoint = os.environ.get("S3_ENDPOINT", "http://seaweedfs-s3.seaweedfs.svc.cluster.local:8333")
s3_bucket = os.environ.get("S3_BUCKET", "mlpipeline")
s3_access_key = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
s3_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")
file_uri = os.environ.get("SPARK_JOB_URI", f"s3a://{s3_bucket}/spark_job.py")

s3_spark_conf = {
    "spark.jars.ivy": "/tmp/.ivy2",
    "spark.driver.extraJavaOptions": "-Divy.home=/tmp/.ivy2",
    "spark.executor.extraJavaOptions": "-Divy.home=/tmp/.ivy2",
    "spark.jars.packages": "org.apache.hadoop:hadoop-aws:3.4.1,software.amazon.awssdk:s3:2.24.6,software.amazon.awssdk:s3-transfer-manager:2.24.6",
    "spark.jars.excludes": "software.amazon.awssdk:bundle",
    "spark.hadoop.fs.s3a.endpoint": s3_endpoint,
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.hadoop.fs.s3a.connection.timeout": "10000",
    "spark.hadoop.fs.s3a.endpoint.region": "us-east-1",
}

if s3_access_key and s3_secret_key:
    s3_spark_conf["spark.hadoop.fs.s3a.access.key"] = s3_access_key
    s3_spark_conf["spark.hadoop.fs.s3a.secret.key"] = s3_secret_key
    s3_spark_conf["spark.hadoop.fs.s3a.aws.credentials.provider"] = (
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
else:
    s3_spark_conf["spark.hadoop.fs.s3a.aws.credentials.provider"] = (
        "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider"
    )

print(f"Target script URI: {file_uri}")
print(f"S3A Endpoint: {s3_endpoint}")

## 3. Submit the S3A Batch Job

Submit the `FileJob` using the configured `s3a://` URI.

In [ ]:
job_name = client.submit_job(
    job=FileJob(
        file_source=file_uri,
        args=["10"],
    ),
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512Mi",
    },
    spark_conf=s3_spark_conf,
)

print(f"Job submitted successfully: {job_name}")

## 4. Wait for Job Completion

Poll the job status until it transitions to `SparkJobStatus.COMPLETED`.

In [ ]:
print(f"Waiting for {job_name} to complete execution...")
job = client.wait_for_job_status(
    job_name,
    status={SparkJobStatus.COMPLETED},
    timeout=600,
)

assert job is not None, f"Job {job_name} timed out or failed to return status"
assert job.status == SparkJobStatus.COMPLETED, f"Job failed with status: {job.status}"

print(f"Job {job_name} completed successfully! Driver Pod: {job.driver_pod_name}")

## 5. Retrieve Job Metadata

Fetch and verify the completed job's metadata.

In [ ]:
job_info = client.get_job(job_name)
assert job_info is not None, f"Failed to retrieve job metadata for {job_name}"
print(f"Retrieved Job Metadata: Name={job_info.name}, Status={job_info.status}")

## 6. Stream Driver Logs

Read and display the driver pod execution output.

In [ ]:
print(f"Driver pod logs for {job_name}:")
print("-" * 70)
for line in client.get_job_logs(job_name):
    print(line.rstrip())
print("-" * 70)

## 7. Clean Up Resources

Delete the completed Spark job.

In [ ]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Cleanup complete.")